In [2]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../data/processed")

files = [
    "clean_sales.parquet",
    "clean_transactions.parquet",
    "daily_demand.parquet",
]

for filename in files:
    path = DATA_DIR / filename

    print("\n" + "=" * 75)
    print(f"FILE: {path}")
    print("=" * 75)

    if not path.exists():
        print("Not found")
        continue

    df = pd.read_parquet(path)

    print("Shape:", df.shape)
    print("\nColumns:")
    print(df.columns.tolist())

    print("\nData types:")
    print(df.dtypes.to_string())

    print("\nFirst 5 rows:")
    print(df.head().to_string(index=False))

    print("\nMissing values:")
    print(df.isna().sum().to_string())

    print("\nDuplicate rows:", df.duplicated().sum())



FILE: ..\data\processed\clean_sales.parquet
Shape: (1007913, 11)

Columns:
['invoice', 'stock_code', 'description', 'quantity', 'invoice_date', 'unit_price', 'customer_id', 'country', 'is_cancelled', 'revenue', 'description_clean']

Data types:
invoice                      string
stock_code                   string
description                  string
quantity                      int64
invoice_date         datetime64[us]
unit_price                  float64
customer_id                   Int64
country                      string
is_cancelled                   bool
revenue                     float64
description_clean            string

First 5 rows:
invoice stock_code                         description  quantity        invoice_date  unit_price  customer_id        country  is_cancelled  revenue                   description_clean
 489434      85048 15CM CHRISTMAS GLASS BALL 20 LIGHTS        12 2009-12-01 07:45:00        6.95        13085 United Kingdom         False     83.4 15CM CHRIST

In [3]:

from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path("../data/processed")
sales_path = DATA_DIR / "clean_sales.parquet"

sales = pd.read_parquet(sales_path)

print("=" * 70)
print("1. DATA QUALITY")
print("=" * 70)

print("Rows:", len(sales))
print("Unique products:", sales["stock_code"].nunique())
print("Unique invoices:", sales["invoice"].nunique())
print("Unique customers:", sales["customer_id"].nunique())
print("Date range:", sales["invoice_date"].min(), "to",
      sales["invoice_date"].max())

print("\nCancellation distribution:")
print(sales["is_cancelled"].value_counts(dropna=False))

print("\n" + "=" * 70)
print("2. QUANTITY AND PRICE")
print("=" * 70)

print(sales[["quantity", "unit_price", "revenue"]].describe())

print("\nNegative quantity rows:",
      int((sales["quantity"] < 0).sum()))
print("Zero quantity rows:",
      int((sales["quantity"] == 0).sum()))
print("Zero price rows:",
      int((sales["unit_price"] == 0).sum()))

print("\n" + "=" * 70)
print("3. PRODUCT-LEVEL MOVEMENT")
print("=" * 70)

valid_sales = sales.loc[
    (~sales["is_cancelled"])
    & (sales["quantity"] > 0)
    & (sales["unit_price"] > 0)
].copy()

product_summary = (
    valid_sales.groupby(
        ["stock_code", "description_clean"],
        dropna=False
    )
    .agg(
        Units_Sold=("quantity", "sum"),
        Revenue=("revenue", "sum"),
        Invoice_Count=("invoice", "nunique"),
        Active_Days=("invoice_date",
                     lambda x: x.dt.date.nunique()),
        First_Sale=("invoice_date", "min"),
        Last_Sale=("invoice_date", "max"),
    )
    .reset_index()
    .sort_values("Units_Sold", ascending=False)
)

print("Valid sales rows:", len(valid_sales))
print("Products:", len(product_summary))

print("\nTop 15 products by units sold:")
print(product_summary.head(15).to_string(index=False))

print("\nBottom 15 products by units sold:")
print(
    product_summary.tail(15)
    .sort_values("Units_Sold")
    .to_string(index=False)
)

print("\n" + "=" * 70)
print("4. INVENTORY INPUT AVAILABILITY")
print("=" * 70)

inventory_fields = [
    "stock_on_hand",
    "inventory_quantity",
    "lead_time",
    "supplier",
    "reorder_point",
    "safety_stock",
]

all_columns = set(sales.columns)

for filename in [
    "clean_transactions.parquet",
    "clean_returns.parquet",
    "clean_cancelled.parquet",
]:
    path = DATA_DIR / filename
    if path.exists():
        df_columns = set(pd.read_parquet(path).columns)
        all_columns.update(df_columns)
        print(f"{filename} columns:", sorted(df_columns))

print("\nPotential inventory fields found:")
print([c for c in inventory_fields if c in all_columns])

print("\nInspection completed.")


1. DATA QUALITY
Rows: 1007913
Unique products: 4917
Unique invoices: 40077
Unique customers: 5878
Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00

Cancellation distribution:
is_cancelled
False    1007913
Name: count, dtype: int64

2. QUANTITY AND PRICE
           quantity    unit_price       revenue
count  1.007913e+06  1.007913e+06  1.007913e+06
mean   1.111718e+01  4.074252e+00  2.031550e+01
std    1.284700e+02  5.043045e+01  2.057158e+02
min    1.000000e+00  1.000000e-03  1.000000e-03
25%    1.000000e+00  1.250000e+00  4.130000e+00
50%    4.000000e+00  2.100000e+00  1.008000e+01
75%    1.200000e+01  4.130000e+00  1.770000e+01
max    8.099500e+04  2.511109e+04  1.684696e+05

Negative quantity rows: 0
Zero quantity rows: 0
Zero price rows: 0

3. PRODUCT-LEVEL MOVEMENT
Valid sales rows: 1007913
Products: 5630

Top 15 products by units sold:
stock_code                   description_clean  Units_Sold   Revenue  Invoice_Count  Active_Days          First_Sale           Last_Sale
   

In [4]:
# 1. Locate the RetailPulse project
cwd = Path.cwd()

if cwd.name.lower() == "notebooks":
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

DATA_DIR = PROJECT_ROOT / "data" / "processed"

SALES_PATH = DATA_DIR / "clean_sales.parquet"

print("Project root:", PROJECT_ROOT)
print("Processed data directory:", DATA_DIR)
print("Sales dataset exists:", SALES_PATH.exists())

if not SALES_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {SALES_PATH}. "
        "Check the notebook working directory."
    )


# 2. Load the primary dataset
sales = pd.read_parquet(SALES_PATH)

print("\n" + "=" * 65)
print("SALES DATASET OVERVIEW")
print("=" * 65)

print("Shape:", sales.shape)
print("\nColumns:")
print(sales.columns.tolist())

print("\nData types:")
print(sales.dtypes)

print("\nMissing values:")
print(sales.isna().sum().sort_values(ascending=False).to_string())

print("\nDuplicate rows:", sales.duplicated().sum())

# 3. Inspect transaction and quantity quality
print("\n" + "=" * 65)
print("TRANSACTION QUALITY CHECKS")
print("=" * 65)

print("\nInvoice date range:")
print("Minimum:", sales["invoice_date"].min())
print("Maximum:", sales["invoice_date"].max())

print("\nQuantity statistics:")
print(sales["quantity"].describe().to_string())

print("\nUnit price statistics:")
print(sales["unit_price"].describe().to_string())

print("\nRevenue statistics:")
print(sales["revenue"].describe().to_string())

print("\nCancelled flag distribution:")
print(sales["is_cancelled"].value_counts(dropna=False).to_string())

print("\nQuantity categories:")
print(
    pd.Series({
        "Positive quantity": (sales["quantity"] > 0).sum(),
        "Zero quantity": (sales["quantity"] == 0).sum(),
        "Negative quantity": (sales["quantity"] < 0).sum(),
        "Zero unit price": (sales["unit_price"] == 0).sum(),
        "Negative unit price": (sales["unit_price"] < 0).sum(),
        "Missing stock code": sales["stock_code"].isna().sum(),
        "Missing description": sales["description"].isna().sum(),
        "Missing invoice date": sales["invoice_date"].isna().sum(),
        "Missing invoice ID": sales["invoice"].isna().sum(),
    }).to_string()
)


# 4. Examine cancellation, return, and price patterns
print("\n" + "=" * 65)
print("QUANTITY AND CANCELLATION CROSS-CHECK")
print("=" * 65)

print(
    pd.crosstab(
        sales["is_cancelled"],
        pd.cut(
            sales["quantity"],
            bins=[-np.inf, -0.001, 0, np.inf],
            labels=["Negative", "Zero", "Positive"]
        ),
        dropna=False
    ).to_string()
)

print("\nExamples of negative-quantity records:")
negative_examples = sales.loc[
    sales["quantity"] < 0,
    [
        "invoice", "stock_code", "description",
        "quantity", "invoice_date", "unit_price",
        "is_cancelled", "revenue"
    ]
].head(10)

print(negative_examples.to_string(index=False))

print("\nExamples of zero-price records:")
zero_price_examples = sales.loc[
    sales["unit_price"] == 0,
    [
        "invoice", "stock_code", "description",
        "quantity", "invoice_date", "unit_price",
        "is_cancelled", "revenue"
    ]
].head(10)

print(zero_price_examples.to_string(index=False))


# 5. Build a provisional product movement summary
# This is an exploratory filter, NOT a final business rule.
# Positive quantity, positive price, non-cancelled records
# are used only to inspect ordinary sales activity.

valid_sales = sales.loc[
    (sales["quantity"] > 0)
    & (sales["unit_price"] > 0)
    & (sales["is_cancelled"] == False)
    & sales["stock_code"].notna()
    & sales["invoice_date"].notna()
].copy()

print("\n" + "=" * 65)
print("PROVISIONAL SALES FILTER")
print("=" * 65)

print("Original rows:", len(sales))
print("Rows after provisional filter:", len(valid_sales))
print("Rows excluded:", len(sales) - len(valid_sales))
print(
    "Distinct products in provisional sales:",
    valid_sales["stock_code"].nunique()
)

# Group by product without assuming that sales equal stock on hand.
product_movement = (
    valid_sales.groupby("stock_code", observed=True)
    .agg(
        description=("description", "first"),
        units_sold=("quantity", "sum"),
        revenue=("revenue", "sum"),
        transaction_count=("invoice", "nunique"),
        active_selling_days=("invoice_date", "nunique"),
        first_sale=("invoice_date", "min"),
        last_sale=("invoice_date", "max"),
        average_unit_price=("unit_price", "mean"),
    )
    .reset_index()
)

product_movement["units_per_active_day"] = (product_movement["units_sold"]/ product_movement["active_selling_days"].replace(0, np.nan))

product_movement = product_movement.sort_values("units_sold", ascending=False)

print("\nProduct movement summary shape:", product_movement.shape)

print("\nTop 15 products by units sold:")
print(product_movement.head(15).to_string(index=False))

print("\nProduct movement summary statistics:")
print(
    product_movement[
        [
            "units_sold",
            "revenue",
            "transaction_count",
            "active_selling_days",
            "units_per_active_day",
        ]
    ].describe().to_string()
)


# 6. Inspect all available processed dataset schemas
print("\n" + "=" * 65)
print("PROCESSED DATASET SCHEMA INVENTORY")
print("=" * 65)

inventory_keywords = [
    "stock", "inventory", "on_hand", "onhand",
    "supplier", "lead_time", "leadtime",
    "purchase", "procurement", "reorder",
    "warehouse", "receipt", "delivery",
    "opening_balance", "closing_balance",
    "stockout", "availability", "cost"
]

schema_rows = []

for path in sorted(DATA_DIR.iterdir()):
    if not path.is_file():
        continue

    suffix = path.suffix.lower()

    if suffix == ".parquet":
        try:
            import pyarrow.parquet as pq

            schema = pq.read_schema(path)
            columns = schema.names

            schema_rows.append({
                "file": path.name,
                "format": "Parquet",
                "columns": columns,
                "size_mb": round(path.stat().st_size / (1024 ** 2), 2)
            })

        except Exception as exc:
            print(f"Could not inspect {path.name}: {exc}")

    elif suffix == ".csv":
        try:
            columns = pd.read_csv(path, nrows=0).columns.tolist()

            schema_rows.append({
                "file": path.name,
                "format": "CSV",
                "columns": columns,
                "size_mb": round(path.stat().st_size / (1024 ** 2), 2)
            })

        except Exception as exc:
            print(f"Could not inspect {path.name}: {exc}")

for item in schema_rows:
    print(f"\nFile: {item['file']} ({item['format']})")
    print("Size (MB):", item["size_mb"])
    print("Columns:", item["columns"])

    matched = [
        col for col in item["columns"]
        if any(keyword in col.lower() for keyword in inventory_keywords)
    ]

    print("Potential inventory-related columns:", matched)


# 7. Summarize what the available schemas suggest
all_columns = {
    (item["file"], col.lower())
    for item in schema_rows
    for col in item["columns"]
}

print("\n" + "=" * 65)
print("INITIAL INVENTORY DATA AVAILABILITY")
print("=" * 65)

for term in [
    "stock_on_hand", "current_stock", "quantity_on_hand",
    "supplier", "lead_time", "purchase_order",
    "stockout", "unit_cost"
]:
    matches = [
        (filename, column)
        for filename, column in sorted(all_columns)
        if term in column
    ]

    print(f"{term}: {matches if matches else 'No matching column found'}")

print("\nInspection completed. No files were written or modified.")


Project root: e:\RetailPulse
Processed data directory: e:\RetailPulse\data\processed
Sales dataset exists: True

SALES DATASET OVERVIEW
Shape: (1007913, 11)

Columns:
['invoice', 'stock_code', 'description', 'quantity', 'invoice_date', 'unit_price', 'customer_id', 'country', 'is_cancelled', 'revenue', 'description_clean']

Data types:
invoice                      string
stock_code                   string
description                  string
quantity                      int64
invoice_date         datetime64[us]
unit_price                  float64
customer_id                   Int64
country                      string
is_cancelled                   bool
revenue                     float64
description_clean            string
dtype: object

Missing values:
customer_id          228488
stock_code                0
invoice                   0
description               0
quantity                  0
invoice_date              0
unit_price                0
country                   0
is_cancelled

In [5]:
# RetailPulse — Inventory Inspection Summary
# Run after the previous inspection cell


print("=" * 60)
print("1. DATA QUALITY SUMMARY")
print("=" * 60)

print("Dataset shape:", sales.shape)
print("Duplicate rows:", sales.duplicated().sum())
print("Date range:", sales["invoice_date"].min(), "to",
      sales["invoice_date"].max())

print("\nMissing values:")
print(sales.isna().sum().to_string())

print("\nCancellation distribution:")
print(sales["is_cancelled"].value_counts(dropna=False).to_dict())

print("\nQuantity and price checks:")
print({
    "positive_quantity": int((sales["quantity"] > 0).sum()),
    "zero_quantity": int((sales["quantity"] == 0).sum()),
    "negative_quantity": int((sales["quantity"] < 0).sum()),
    "zero_price": int((sales["unit_price"] == 0).sum()),
    "negative_price": int((sales["unit_price"] < 0).sum()),
})

print("\nCancellation status by quantity:")
print(pd.crosstab(
        sales["is_cancelled"],
        pd.cut(
            sales["quantity"],
            bins=[-np.inf, -0.001, 0, np.inf],
            labels=["Negative", "Zero", "Positive"]
        )
    ).to_string()
)

print("\n" + "=" * 60)
print("2. PROVISIONAL PRODUCT SALES")
print("=" * 60)

print("Filtered sales rows:", len(valid_sales))
print("Excluded rows:", len(sales) - len(valid_sales))
print("Distinct products:", valid_sales["stock_code"].nunique())
print("Product summary shape:", product_movement.shape)

print("\nTop 10 products by units sold:")
print(
    product_movement[
        [
            "stock_code",
            "description",
            "units_sold",
            "revenue",
            "transaction_count",
            "active_selling_days",
        ]
    ].head(10).to_string(index=False)
)

print("\n" + "=" * 60)
print("3. INVENTORY-RELATED DATASET SCHEMAS")
print("=" * 60)

for item in schema_rows:
    print(f"\n{item['file']}")
    print(", ".join(item["columns"]))

    matched = [
        col for col in item["columns"]
        if any(
            keyword in col.lower()
            for keyword in [
                "stock", "inventory", "supplier", "lead_time",
                "purchase", "reorder", "warehouse", "receipt",
                "stockout", "on_hand", "onhand", "unit_cost"
            ]
        )
    ]

    if matched:
        print("Potential inventory fields:", matched)

print("\nInspection summary complete.")


1. DATA QUALITY SUMMARY
Dataset shape: (1007913, 11)
Duplicate rows: 0
Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00

Missing values:
invoice                   0
stock_code                0
description               0
quantity                  0
invoice_date              0
unit_price                0
customer_id          228488
country                   0
is_cancelled              0
revenue                   0
description_clean         0

Cancellation distribution:
{False: 1007913}

Quantity and price checks:
{'positive_quantity': 1007913, 'zero_quantity': 0, 'negative_quantity': 0, 'zero_price': 0, 'negative_price': 0}

Cancellation status by quantity:
quantity      Positive
is_cancelled          
False          1007913

2. PROVISIONAL PRODUCT SALES
Filtered sales rows: 1007913
Excluded rows: 0
Distinct products: 4917
Product summary shape: (4917, 10)

Top 10 products by units sold:
stock_code                        description  units_sold   revenue  transaction_count  active

In [6]:
DATA_DIR = PROJECT_ROOT / "data" / "processed"

print("=" * 65)
print("1. CHECK SALES DATA CONSISTENCY")
print("=" * 65)

# Verify whether revenue is consistent with quantity * unit_price.
sales_check = sales[["quantity", "unit_price", "revenue"]].copy()

sales_check["calculated_revenue"] = (sales_check["quantity"] * sales_check["unit_price"])

sales_check["revenue_difference"] = (sales_check["revenue"] - sales_check["calculated_revenue"]).abs()

print(
    "Rows where revenue differs from quantity * unit_price "
    "by more than 0.01:",
    int((sales_check["revenue_difference"] > 0.01).sum())
)

print(
    "Maximum absolute revenue difference:",
    round(sales_check["revenue_difference"].max(), 6)
)

print("\n" + "=" * 65)
print("2. INSPECT RETURNS AND CANCELLATIONS")
print("=" * 65)

for filename in [
    "clean_returns.parquet",
    "clean_cancelled.parquet",
    "clean_transactions.parquet",
]:
    path = DATA_DIR / filename

    print(f"\n--- {filename} ---")

    if not path.exists():
        print("File not found.")
        continue

    # Read schema and a small sample first.
    if path.suffix == ".parquet":
        import pyarrow.parquet as pq

        parquet_file = pq.ParquetFile(path)
        print("Rows:", parquet_file.metadata.num_rows)
        print("Columns:", parquet_file.schema_arrow.names)

        sample = pd.read_parquet(path).head(5)

    print("\nSample records:")
    print(sample.to_string(index=False))

    # For the two dedicated exception datasets, inspect their
    # full quantity and cancellation distributions.
    if filename in [
        "clean_returns.parquet",
        "clean_cancelled.parquet",
    ]:
        df = pd.read_parquet(path)

        print("\nShape:", df.shape)

        if "quantity" in df.columns:
            print(
                "Quantity distribution:",
                {
                    "negative": int((df["quantity"] < 0).sum()),
                    "zero": int((df["quantity"] == 0).sum()),
                    "positive": int((df["quantity"] > 0).sum()),
                }
            )

        if "is_cancelled" in df.columns:
            print(
                "Cancellation distribution:",
                df["is_cancelled"].value_counts(
                    dropna=False
                ).to_dict()
            )

print("\n" + "=" * 65)
print("3. PRODUCT IDENTIFIER QUALITY")
print("=" * 65)

print("Distinct stock codes:", sales["stock_code"].nunique())
print("Distinct descriptions:", sales["description"].nunique())
print(
    "Products with multiple descriptions:",
    int(
        (
            sales.groupby("stock_code")["description"]
            .nunique()
            .gt(1)
        ).sum()
    )
)

print("\nInspection complete. No existing project files were modified.")


1. CHECK SALES DATA CONSISTENCY
Rows where revenue differs from quantity * unit_price by more than 0.01: 0
Maximum absolute revenue difference: 0.0

2. INSPECT RETURNS AND CANCELLATIONS

--- clean_returns.parquet ---
Rows: 22496
Columns: ['invoice', 'stock_code', 'description', 'quantity', 'invoice_date', 'unit_price', 'customer_id', 'country', 'is_cancelled', 'revenue', 'description_clean']

Sample records:
invoice stock_code                   description  quantity        invoice_date  unit_price  customer_id   country  is_cancelled  revenue             description_clean
C489449      22087      PAPER BUNTING WHITE LACE       -12 2009-12-01 10:33:00        2.95        16321 Australia          True    -35.4      PAPER BUNTING WHITE LACE
C489449     85206A  CREAM FELT EASTER EGG BASKET        -6 2009-12-01 10:33:00        1.65        16321 Australia          True     -9.9  CREAM FELT EASTER EGG BASKET
C489449      21895 POTTING SHED SOW 'N' GROW SET        -4 2009-12-01 10:33:00        4

In [7]:
# RetailPulse — Returns and Cancellation Validation

returns = pd.read_parquet(DATA_DIR / "clean_returns.parquet")
cancelled = pd.read_parquet(DATA_DIR / "clean_cancelled.parquet")

print("=" * 60)
print("1. DATASET SIZES")
print("=" * 60)

print("Sales:", sales.shape)
print("Returns:", returns.shape)
print("Cancelled:", cancelled.shape)

print("\n" + "=" * 60)
print("2. CANCELLATION DATASET")
print("=" * 60)

print("Columns:", cancelled.columns.tolist())
print("\nCancellation flag:")
print(cancelled["is_cancelled"].value_counts(dropna=False))

print("\nQuantity distribution:")
print({
    "negative": int((cancelled["quantity"] < 0).sum()),
    "zero": int((cancelled["quantity"] == 0).sum()),
    "positive": int((cancelled["quantity"] > 0).sum()),
})

print("\nSample cancelled records:")
print(cancelled.head(5).to_string(index=False))

print("\n" + "=" * 60)
print("3. INVOICE OVERLAP")
print("=" * 60)

sales_invoices = set(sales["invoice"].dropna().unique())
return_invoices = set(returns["invoice"].dropna().unique())
cancelled_invoices = set(cancelled["invoice"].dropna().unique())

print("Unique sales invoices:", len(sales_invoices))
print("Unique return invoices:", len(return_invoices))
print("Unique cancelled invoices:", len(cancelled_invoices))

print("Sales/returns shared invoice IDs:",len(sales_invoices & return_invoices))

print("Sales/cancelled shared invoice IDs:",len(sales_invoices & cancelled_invoices))

print("Returns/cancelled shared invoice IDs:",len(return_invoices & cancelled_invoices))

print("\n" + "=" * 60)
print("4. RETURNS FLAG BREAKDOWN")
print("=" * 60)

print(returns.groupby("is_cancelled", dropna=False)
    .agg(
        rows=("invoice", "size"),
        unique_invoices=("invoice", "nunique"),
        total_quantity=("quantity", "sum"),
        total_revenue=("revenue", "sum"),
    )
    .to_string()
)

print("\n" + "=" * 60)
print("5. DATE RANGES")
print("=" * 60)

for name, df in [
    ("Sales", sales),
    ("Returns", returns),
    ("Cancelled", cancelled),
]:
    print(
        f"{name}:",
        df["invoice_date"].min(),
        "to",
        df["invoice_date"].max()
    )

print("\nValidation complete. No files were modified.")


1. DATASET SIZES
Sales: (1007913, 11)
Returns: (22496, 11)
Cancelled: (19104, 11)

2. CANCELLATION DATASET
Columns: ['invoice', 'stock_code', 'description', 'quantity', 'invoice_date', 'unit_price', 'customer_id', 'country', 'is_cancelled', 'revenue', 'description_clean']

Cancellation flag:
is_cancelled
True    19104
Name: count, dtype: int64

Quantity distribution:
{'negative': 19103, 'zero': 0, 'positive': 1}

Sample cancelled records:
invoice stock_code                   description  quantity        invoice_date  unit_price  customer_id   country  is_cancelled  revenue             description_clean
C489449      22087      PAPER BUNTING WHITE LACE       -12 2009-12-01 10:33:00        2.95        16321 Australia          True    -35.4      PAPER BUNTING WHITE LACE
C489449     85206A  CREAM FELT EASTER EGG BASKET        -6 2009-12-01 10:33:00        1.65        16321 Australia          True     -9.9  CREAM FELT EASTER EGG BASKET
C489449      21895 POTTING SHED SOW 'N' GROW SET        

In [8]:
# RetailPulse — Step 5: Daily Product-Demand Profiles
# 1. Define the validated gross-sales inclusion rules

demand_sales = sales.loc[
    (sales["quantity"] > 0)
    & (sales["unit_price"] > 0)
    & (~sales["is_cancelled"])
].copy()

# Use stock_code as the stable product identifier.
# Normalize timestamps to calendar dates for daily aggregation.
demand_sales["sales_date"] = (demand_sales["invoice_date"].dt.normalize())

print("=" * 65)
print("1. VALIDATED SALES INPUT")
print("=" * 65)

print("Input sales rows:", len(sales))
print("Included sales rows:", len(demand_sales))
print("Excluded sales rows:", len(sales) - len(demand_sales))
print("Distinct products:", demand_sales["stock_code"].nunique())
print("Distinct invoices:", demand_sales["invoice"].nunique())


# 2. Aggregate product demand by calendar day
product_daily_demand = (
    demand_sales.groupby(
        ["stock_code", "sales_date"],
        observed=True,
        as_index=False
    )
    .agg(
        daily_units_sold=("quantity", "sum"),
        daily_revenue=("revenue", "sum"),
        daily_invoice_count=("invoice", "nunique")
    )
)

print("\n" + "=" * 65)
print("2. DAILY DEMAND DATASET")
print("=" * 65)

print("Shape:", product_daily_demand.shape)
print("\nColumns:", product_daily_demand.columns.tolist())
print("\nData types:")
print(product_daily_demand.dtypes)

print("\nSample records:")
print(product_daily_demand.head(10).to_string(index=False))

print("\nDate range:")
print(product_daily_demand["sales_date"].min(),"to",product_daily_demand["sales_date"].max())

print(
    "\nDuplicate product-date combinations:",
    product_daily_demand.duplicated(["stock_code", "sales_date"]).sum()
)


# 3. Create product-level movement metrics
product_metadata = (demand_sales.sort_values("invoice_date")
    .groupby("stock_code", observed=True)
    .agg(
        description=("description", "last"),
        description_clean=("description_clean", "last")
    )
    .reset_index()
)

product_movement = (
    product_daily_demand.groupby(
        "stock_code",
        observed=True,
        as_index=False
    )
    .agg(
        units_sold=("daily_units_sold", "sum"),
        revenue=("daily_revenue", "sum"),
        transaction_count=("daily_invoice_count", "sum"),
        active_selling_days=("sales_date", "nunique"),
        first_sale=("sales_date", "min"),
        last_sale=("sales_date", "max")
    )
    .merge(
        product_metadata,
        on="stock_code",
        how="left",
        validate="one_to_one"
    )
)

# Average volume on days with recorded sales.
product_movement["units_per_active_day"] = (product_movement["units_sold"]/ product_movement["active_selling_days"])

# Share of the overall observed calendar period with sales.
period_start = product_daily_demand["sales_date"].min()
period_end = product_daily_demand["sales_date"].max()

calendar_days = ((period_end - period_start).days + 1)

product_movement["observed_calendar_days"] = calendar_days

product_movement["selling_day_rate"] = (product_movement["active_selling_days"]/ calendar_days)

product_movement = product_movement.sort_values("units_sold",ascending=False).reset_index(drop=True)

print("\n" + "=" * 65)
print("3. PRODUCT MOVEMENT SUMMARY")
print("=" * 65)

print("Products:", len(product_movement))
print("Observed calendar days:", calendar_days)

print("\nSummary statistics:")
print(product_movement[
        [
            "units_sold",
            "revenue",
            "transaction_count",
            "active_selling_days",
            "units_per_active_day",
            "selling_day_rate"
        ]
    ].describe().round(3).to_string()
)

print("\nTop 15 products by units sold:")
print(product_movement[
        [
            "stock_code",
            "description",
            "units_sold",
            "revenue",
            "transaction_count",
            "active_selling_days",
            "units_per_active_day",
            "selling_day_rate"
        ]
    ].head(15).round(3).to_string(index=False)
)


# 4. Validate aggregate totals
print("\n" + "=" * 65)
print("4. AGGREGATION VALIDATION")
print("=" * 65)

print("Sales units:", int(demand_sales["quantity"].sum()))
print("Daily aggregated units:",int(product_daily_demand["daily_units_sold"].sum()))

print("Sales revenue:", round(demand_sales["revenue"].sum(), 2))
print("Daily aggregated revenue:",round(product_daily_demand["daily_revenue"].sum(), 2))

print("Product-summary units:", int(product_movement["units_sold"].sum()))

print("Product-summary revenue:", round(product_movement["revenue"].sum(), 2))

print("\nNo files saved yet. Review the results before proceeding.")


1. VALIDATED SALES INPUT
Input sales rows: 1007913
Included sales rows: 1007913
Excluded sales rows: 0
Distinct products: 4917
Distinct invoices: 40077

2. DAILY DEMAND DATASET
Shape: (532839, 5)

Columns: ['stock_code', 'sales_date', 'daily_units_sold', 'daily_revenue', 'daily_invoice_count']

Data types:
stock_code                     string
sales_date             datetime64[us]
daily_units_sold                int64
daily_revenue                 float64
daily_invoice_count             int64
dtype: object

Sample records:
stock_code sales_date  daily_units_sold  daily_revenue  daily_invoice_count
     10002 2009-12-01                12          10.20                    1
     10002 2009-12-03                 6           5.10                    3
     10002 2009-12-04                73          62.05                    4
     10002 2009-12-06                49          41.65                    2
     10002 2009-12-07                 2           1.70                    1
     10002 2009

In [9]:
# RetailPulse — Step 6: Investigate Demand Outliers

print("=" * 70)
print("1. PRODUCTS WITH THE HIGHEST UNITS PER ACTIVE SELLING DAY")
print("=" * 70)

print(product_movement[
        [
            "stock_code",
            "description",
            "units_sold",
            "active_selling_days",
            "units_per_active_day",
            "transaction_count",
        ]
    ]
    .sort_values("units_per_active_day", ascending=False)
    .head(15)
    .to_string(index=False)
)


# 2. Inspect the largest individual product-day observations
print("\n" + "=" * 70)
print("2. LARGEST PRODUCT-DAY DEMAND OBSERVATIONS")
print("=" * 70)

print(product_daily_demand.sort_values("daily_units_sold", ascending=False).head(20).to_string(index=False))


# 3. Investigate product code 23843
target_code = "23843"

print("\n" + "=" * 70)
print(f"3. INVESTIGATE PRODUCT {target_code}")
print("=" * 70)

target_sales = demand_sales.loc[
    demand_sales["stock_code"].astype(str) == target_code,
    [
        "invoice",
        "stock_code",
        "description",
        "quantity",
        "invoice_date",
        "unit_price",
        "customer_id",
        "country",
        "revenue",
    ]
].copy()

print("Raw sales rows:", len(target_sales))
print("Unique invoices:", target_sales["invoice"].nunique())

if not target_sales.empty:
    print("\nQuantity distribution:")
    print(target_sales["quantity"].describe().to_string())

    print("\nLargest individual transactions:")
    print(
        target_sales
        .sort_values("quantity", ascending=False)
        .head(20)
        .to_string(index=False)
    )

    print("\nProduct descriptions:")
    print(
        target_sales.groupby("description", dropna=False)
        .agg(
            rows=("quantity", "size"),
            units=("quantity", "sum"),
            revenue=("revenue", "sum"),
            invoices=("invoice", "nunique"),
        )
        .to_string()
    )


# 4. Compare large transactions with the overall distribution

print("\n" + "=" * 70)
print("4. TRANSACTION QUANTITY EXTREMES")
print("=" * 70)

print("Quantity percentiles:")
print(demand_sales["quantity"].quantile([0.50, 0.75, 0.90, 0.95, 0.99, 0.995, 0.999]).to_string())

print("\nLargest individual quantity records:")
print(demand_sales[
        [
            "invoice",
            "stock_code",
            "description",
            "quantity",
            "invoice_date",
            "unit_price",
            "revenue",
        ]
    ]
    .nlargest(15, "quantity").to_string(index=False)
)

print("\nInvestigation complete. No records were removed.")


1. PRODUCTS WITH THE HIGHEST UNITS PER ACTIVE SELLING DAY
stock_code                        description  units_sold  active_selling_days  units_per_active_day  transaction_count
     23843        PAPER CRAFT , LITTLE BIRDIE       80995                    1          80995.000000                  1
     85220    SMALL FAIRY CAKE FRIDGE MAGNETS       12236                    5           2447.200000                  5
     16044           POP-ART FLUORESCENT PENS        6193                    3           2064.333333                  3
     37410 BLACK AND WHITE PAISLEY FLOWER MUG       25707                   25           1028.280000                 33
     85218  S/5 MINI ICE CREAM FRIDGE MAGNETS        1992                    2            996.000000                  2
     37352                  BIRD IN TREE MUG         4643                    7            663.285714                  8
     23166     MEDIUM CERAMIC TOP STORAGE JAR       78033                  125            624.264000  

In [10]:
# Demand Variability and Intermittency

print("=" * 70)
print("1. BUILD PRODUCT DEMAND PROFILES")
print("=" * 70)

# Use the product-level sales dates already validated.
# For each product, include zero-sales days between its first
# and last observed sale. Do not assume its entire lifecycle
# outside this interval is known.

demand_profile_rows = []

for stock_code, group in product_daily_demand.groupby(
    "stock_code",
    sort=False,
    observed=True
):
    group = group.sort_values("sales_date")

    first_sale = group["sales_date"].min()
    last_sale = group["sales_date"].max()

    # Complete daily series only within observed sales lifespan.
    daily_units = (group.set_index("sales_date")["daily_units_sold"].asfreq("D", fill_value=0))

    total_units = int(daily_units.sum())
    active_days = int((daily_units > 0).sum())
    observed_days = len(daily_units)
    zero_sales_days = observed_days - active_days

    mean_daily = float(daily_units.mean())
    std_daily = float(daily_units.std(ddof=1)) if observed_days > 1 else 0.0

    # CV is undefined when average demand is zero.
    cv_demand = (
        std_daily / mean_daily
        if mean_daily > 0
        else np.nan
    )

    # Inter-demand interval: average days between observed
    # selling days, based on the number of days in the span.
    demand_occurrence_rate = (
        active_days / observed_days
        if observed_days > 0
        else np.nan
    )

    demand_profile_rows.append({
        "stock_code": stock_code,
        "first_sale": first_sale,
        "last_sale": last_sale,
        "observed_lifespan_days": observed_days,
        "active_selling_days": active_days,
        "zero_sales_days": zero_sales_days,
        "units_sold": total_units,
        "mean_daily_demand": mean_daily,
        "std_daily_demand": std_daily,
        "cv_demand": cv_demand,
        "demand_occurrence_rate": demand_occurrence_rate,
        "median_daily_demand": float(daily_units.median()),
        "p95_daily_demand": float(daily_units.quantile(0.95)),
        "max_daily_demand": int(daily_units.max()),
    })

product_demand_profiles = pd.DataFrame(demand_profile_rows)

print("Products profiled:", len(product_demand_profiles))
print("Profile shape:", product_demand_profiles.shape)
print("\nMissing values:")
print(product_demand_profiles.isna().sum().to_string())

# 2. Validate profile totals and date spans
print("\n" + "=" * 70)
print("2. VALIDATION")
print("=" * 70)

print("Units in daily demand:", int(product_daily_demand["daily_units_sold"].sum()))

print("Units in demand profiles:",int(product_demand_profiles["units_sold"].sum()))

print("Products with one active selling day:", int((product_demand_profiles["active_selling_days"] == 1).sum()))

print("Products with no zero-sales days within their observed span:",
    int((product_demand_profiles["zero_sales_days"] == 0).sum())
)


# 3. Inspect demand variability distribution
print("\n" + "=" * 70)
print("3. DEMAND PROFILE SUMMARY")
print("=" * 70)

profile_columns = [
    "observed_lifespan_days",
    "active_selling_days",
    "zero_sales_days",
    "mean_daily_demand",
    "std_daily_demand",
    "cv_demand",
    "demand_occurrence_rate",
    "median_daily_demand",
    "p95_daily_demand",
    "max_daily_demand",
]

print(
    product_demand_profiles[profile_columns]
    .describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
    .round(3)
    .to_string()
)

# 4. Inspect the most intermittent products
print("\n" + "=" * 70)
print("4. PRODUCTS WITH LOWEST DEMAND OCCURRENCE")
print("=" * 70)

intermittent_view = product_demand_profiles.merge(
    product_movement[["stock_code", "description", "revenue", "transaction_count"]],
    on="stock_code",
    how="left",
    validate="one_to_one"
)

print(intermittent_view[
        [
            "stock_code",
            "description",
            "units_sold",
            "observed_lifespan_days",
            "active_selling_days",
            "zero_sales_days",
            "demand_occurrence_rate",
            "mean_daily_demand",
            "cv_demand",
        ]
    ]
    .sort_values(["demand_occurrence_rate", "units_sold"],ascending=[True, False])
    .head(15).round(3).to_string(index=False)
)


# 5. Inspect high-variability products
print("\n" + "=" * 70)
print("5. PRODUCTS WITH HIGHEST DEMAND VARIABILITY")
print("=" * 70)

print(intermittent_view[
        [
            "stock_code",
            "description",
            "units_sold",
            "active_selling_days",
            "mean_daily_demand",
            "std_daily_demand",
            "cv_demand",
            "max_daily_demand",
        ]
    ]
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=["cv_demand"])
    .sort_values("cv_demand", ascending=False)
    .head(15)
    .round(3)
    .to_string(index=False)
)

print("\nNo files saved. Classification thresholds have not been chosen.")


1. BUILD PRODUCT DEMAND PROFILES
Products profiled: 4917
Profile shape: (4917, 14)

Missing values:
stock_code                0
first_sale                0
last_sale                 0
observed_lifespan_days    0
active_selling_days       0
zero_sales_days           0
units_sold                0
mean_daily_demand         0
std_daily_demand          0
cv_demand                 0
demand_occurrence_rate    0
median_daily_demand       0
p95_daily_demand          0
max_daily_demand          0

2. VALIDATION
Units in daily demand: 11205148
Units in demand profiles: 11205148
Products with one active selling day: 177
Products with no zero-sales days within their observed span: 184

3. DEMAND PROFILE SUMMARY
       observed_lifespan_days  active_selling_days  zero_sales_days  mean_daily_demand  std_daily_demand  cv_demand  demand_occurrence_rate  median_daily_demand  p95_daily_demand  max_daily_demand
count                4917.000             4917.000         4917.000           4917.000         

In [11]:
#Inspect Product Movement Thresholds
# Reuse the existing demand profile DataFrame.
# If your notebook uses a different variable name, assign it here.
profiles = product_demand_profiles.copy()

print("=" * 65)
print("1. DEMAND OCCURRENCE RATE DISTRIBUTION")
print("=" * 65)

print(
    profiles["demand_occurrence_rate"]
    .describe(
        percentiles=[
            0.10, 0.20, 0.25, 0.33, 0.50,
            0.67, 0.75, 0.80, 0.90, 0.95
        ]
    )
    .round(4).to_string()
)

print("\n" + "=" * 65)
print("2. CANDIDATE DATA-DRIVEN THRESHOLDS")
print("=" * 65)

occurrence_thresholds = (profiles["demand_occurrence_rate"].quantile([1/3, 2/3]))

print("Lower tercile:", round(occurrence_thresholds.iloc[0], 4))
print("Upper tercile:", round(occurrence_thresholds.iloc[1], 4))

print("\n" + "=" * 65)
print("3. PRODUCT COUNTS AT SELECTED OCCURRENCE RATES")
print("=" * 65)

for threshold in [0.05, 0.10, 0.20, 0.30, 0.50, 0.75]:
    count = int((profiles["demand_occurrence_rate"] >= threshold).sum())

    print(
        f"Occurrence rate >= {threshold:.0%}: "
        f"{count:,} products ({count / len(profiles):.1%})"
    )

print("\n" + "=" * 65)
print("4. PRODUCTS WITH EXTREME DEMAND OCCURRENCE")
print("=" * 65)

print("\nHighest occurrence rates:")
print(
    profiles.sort_values("demand_occurrence_rate", ascending=False)[
        [
            "stock_code",
            "observed_lifespan_days",
            "active_selling_days",
            "demand_occurrence_rate",
            "mean_daily_demand",
            "cv_demand",
        ]
    ].head(10).round(3).to_string(index=False)
)

print("\nLowest occurrence rates:")
print(profiles.sort_values("demand_occurrence_rate", ascending=True)[
        [
            "stock_code",
            "observed_lifespan_days",
            "active_selling_days",
            "demand_occurrence_rate",
            "mean_daily_demand",
            "cv_demand",
        ]
    ].head(10).round(3).to_string(index=False)
)

print("\nThreshold inspection complete. No files saved.")


1. DEMAND OCCURRENCE RATE DISTRIBUTION
count    4917.0000
mean        0.2759
std         0.2521
min         0.0041
10%         0.0300
20%         0.0600
25%         0.0753
33%         0.1063
50%         0.1918
67%         0.3302
75%         0.4134
80%         0.4715
90%         0.6429
95%         0.7964
max         1.0000

2. CANDIDATE DATA-DRIVEN THRESHOLDS
Lower tercile: 0.1078
Upper tercile: 0.3277

3. PRODUCT COUNTS AT SELECTED OCCURRENCE RATES
Occurrence rate >= 5%: 4,080 products (83.0%)
Occurrence rate >= 10%: 3,369 products (68.5%)
Occurrence rate >= 20%: 2,398 products (48.8%)
Occurrence rate >= 30%: 1,782 products (36.2%)
Occurrence rate >= 50%: 890 products (18.1%)
Occurrence rate >= 75%: 307 products (6.2%)

4. PRODUCTS WITH EXTREME DEMAND OCCURRENCE

Highest occurrence rates:
  stock_code  observed_lifespan_days  active_selling_days  demand_occurrence_rate  mean_daily_demand  cv_demand
gift_0001_70                       1                    1                     1.0       

In [14]:
print("Columns in product_demand_profiles:")
print(product_demand_profiles.columns.tolist())

print("\nColumns in profiles:")
print(profiles.columns.tolist())


Columns in product_demand_profiles:
['stock_code', 'first_sale', 'last_sale', 'observed_lifespan_days', 'active_selling_days', 'zero_sales_days', 'units_sold', 'mean_daily_demand', 'std_daily_demand', 'cv_demand', 'demand_occurrence_rate', 'median_daily_demand', 'p95_daily_demand', 'max_daily_demand']

Columns in profiles:
['stock_code', 'first_sale', 'last_sale', 'observed_lifespan_days', 'active_selling_days', 'zero_sales_days', 'units_sold', 'mean_daily_demand', 'std_daily_demand', 'cv_demand', 'demand_occurrence_rate', 'median_daily_demand', 'p95_daily_demand', 'max_daily_demand', 'movement_class', 'history_quality', 'classification_reliability']


In [16]:
# Add product descriptions without duplicating existing columns.

if "description" not in profiles.columns:
    description_lookup = (product_movement[["stock_code", "description"]].drop_duplicates(subset=["stock_code"]))

    profiles = profiles.merge(
        description_lookup,
        on="stock_code",
        how="left",
        validate="one_to_one"
    )

print("Profiles shape:", profiles.shape)
print("Description column exists:", "description" in profiles.columns)
print("Missing descriptions:", profiles["description"].isna().sum())
print("Duplicate stock codes:", profiles["stock_code"].duplicated().sum())


Profiles shape: (4917, 19)
Description column exists: True
Missing descriptions: 0
Duplicate stock codes: 0


In [17]:

display_columns = [
    "stock_code",
    "description",
    "units_sold",
    "revenue",
    "active_selling_days",
    "observed_lifespan_days",
    "demand_occurrence_rate",
    "movement_class",
    "history_quality",
]

for movement in ["Fast-moving", "Medium-moving", "Slow-moving"]:
    print(f"\n{movement}:")

    subset = profiles.loc[profiles["movement_class"] == movement].sort_values("units_sold", ascending=False)

    print(subset[display_columns].head(5).round(3).to_string(index=False))

print("\nIntegrity checks:")
print("Products:", len(profiles))
print("Duplicate stock codes:", profiles["stock_code"].duplicated().sum())
print("Missing movement classes:", profiles["movement_class"].isna().sum())
print("Missing history flags:", profiles["history_quality"].isna().sum())

print("Units preserved:",np.isclose(
        profiles["units_sold"].sum(),
        product_demand_profiles["units_sold"].sum()
    )
)

print("Revenue matches product movement:",np.isclose(
        profiles["revenue"].sum(),
        product_movement["revenue"].sum()
    )
)



Fast-moving:
stock_code                        description  units_sold   revenue  active_selling_days  observed_lifespan_days  demand_occurrence_rate movement_class             history_quality
     84077  WORLD WAR 2 GLIDERS ASSTD DESIGNS      106139  24445.61                  470                     739                   0.636    Fast-moving Sufficient observed history
    85099B            JUMBO BAG RED RETROSPOT       96757 180569.34                  589                     739                   0.797    Fast-moving Sufficient observed history
     21212    PACK OF 72 RETROSPOT CAKE CASES       94884  51825.15                  594                     739                   0.804    Fast-moving Sufficient observed history
    85123A CREAM HANGING HEART T-LIGHT HOLDER       94203 257724.71                  603                     739                   0.816    Fast-moving Sufficient observed history
     22197                     POPCORN HOLDER       88499  79520.20                  5

In [18]:
# RetailPulse — Step 12: Save Validated Inventory Outputs

DATA_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# 1. Validate before saving
assert len(profiles) == 4917, "Unexpected product count."
assert profiles["stock_code"].is_unique, "Duplicate product codes."
assert profiles["movement_class"].notna().all(), "Missing movement classes."
assert profiles["history_quality"].notna().all(), "Missing history flags."

assert np.isclose(
    profiles["units_sold"].sum(),
    product_demand_profiles["units_sold"].sum()
), "Unit totals changed."

assert np.isclose(
    profiles["revenue"].sum(),
    product_movement["revenue"].sum()
), "Revenue totals do not match."


# 2. Prepare output tables
profile_output = profiles.copy()

movement_output = profiles.sort_values(["movement_class", "units_sold"],ascending=[True, False]).copy()

priority_summary = (profiles.groupby("movement_class", observed=True).agg(
        product_count=("stock_code", "nunique"),
        total_units=("units_sold", "sum"),
        total_revenue=("revenue", "sum"),
        median_daily_demand=("mean_daily_demand", "median"),
        median_occurrence_rate=("demand_occurrence_rate", "median"),
        limited_history_products=(
            "history_quality",
            lambda x: (x == "Limited observed history").sum()
        ),
    )
    .reset_index()
)

priority_summary["product_share"] = (priority_summary["product_count"] / len(profiles))

priority_summary["unit_share"] = (priority_summary["total_units"] / profiles["units_sold"].sum())

priority_summary["revenue_share"] = (priority_summary["total_revenue"] / profiles["revenue"].sum())


# 3. Define output paths
profile_path = DATA_DIR / "product_demand_profiles.parquet"
movement_path = DATA_DIR / "product_movement_analysis.csv"
summary_path = OUTPUT_DIR / "inventory_priority_summary.csv"

# Prevent accidental overwriting.
for path in [profile_path, movement_path, summary_path]:
    if path.exists():
        raise FileExistsError(f"{path} already exists. Review it before overwriting.")


# 4. Save outputs
profile_output.to_parquet(profile_path, index=False)
movement_output.to_csv(movement_path, index=False)
priority_summary.to_csv(summary_path, index=False)

print("Saved successfully:")
print(profile_path)
print(movement_path)
print(summary_path)

print("\nOutput dimensions:")
print("Demand profiles:", profile_output.shape)
print("Movement analysis:", movement_output.shape)
print("Priority summary:", priority_summary.shape)

print("\nPriority summary:")
print(priority_summary.round(4).to_string(index=False))


# 5. Reload and validate the saved files
saved_profiles = pd.read_parquet(profile_path)
saved_movement = pd.read_csv(movement_path)
saved_summary = pd.read_csv(summary_path)

assert len(saved_profiles) == len(profiles)
assert len(saved_movement) == len(profiles)
assert len(saved_summary) == 3

assert saved_profiles["stock_code"].is_unique
assert saved_movement["stock_code"].is_unique

assert np.isclose(saved_profiles["units_sold"].sum(),profiles["units_sold"].sum())

assert np.isclose(saved_profiles["revenue"].sum(),profiles["revenue"].sum())

print("\nAll saved-file validation checks passed.")


Saved successfully:
e:\RetailPulse\data\processed\product_demand_profiles.parquet
e:\RetailPulse\data\processed\product_movement_analysis.csv
e:\RetailPulse\outputs\inventory_priority_summary.csv

Output dimensions:
Demand profiles: (4917, 19)
Movement analysis: (4917, 19)
Priority summary: (3, 10)

Priority summary:
movement_class  product_count  total_units  total_revenue  median_daily_demand  median_occurrence_rate  limited_history_products  product_share  unit_share  revenue_share
   Fast-moving           1639      8295470   16431078.790               7.1347                  0.5237                       383         0.3333      0.7403         0.8024
 Medium-moving           1639      2317507    3344458.570               1.7145                  0.1918                       125         0.3333      0.2068         0.1633
   Slow-moving           1639       592171     700723.088               0.1819                  0.0486                       233         0.3333      0.0528         0.03

In [19]:
#Recent Demand Trend Analysis

DATA_DIR = PROJECT_ROOT / "data" / "processed"

# Reload the validated daily demand data from existing notebook state.
daily = product_daily_demand.copy()

daily["sales_date"] = pd.to_datetime(daily["sales_date"])

# Anchor the comparison to the last date in the dataset.
# This avoids incorrectly assuming that the data ends today.
data_end = daily["sales_date"].max()

recent_start = data_end - pd.Timedelta(days=89)
previous_start = recent_start - pd.Timedelta(days=90)
previous_end = recent_start - pd.Timedelta(days=1)

print("=" * 70)
print("1. COMPARISON WINDOWS")
print("=" * 70)

print("Dataset end date:", data_end.date())
print("Recent window:", recent_start.date(), "to", data_end.date())
print("Previous window:", previous_start.date(), "to", previous_end.date())


# 1. Aggregate observed sales within each window.
# Products with no sales in a window receive zero units.


recent = (daily.loc[daily["sales_date"].between(recent_start, data_end)]
    .groupby("stock_code", observed=True).agg(
        recent_units=("daily_units_sold", "sum"),
        recent_revenue=("daily_revenue", "sum"),
        recent_active_days=("sales_date", "nunique"),
    )
)

previous = (daily.loc[daily["sales_date"].between(previous_start, previous_end)]
    .groupby("stock_code", observed=True).agg(
        previous_units=("daily_units_sold", "sum"),
        previous_revenue=("daily_revenue", "sum"),
        previous_active_days=("sales_date", "nunique"),
    )
)

trend = (profiles[
        [
            "stock_code",
            "description",
            "movement_class",
            "history_quality",
            "units_sold",
        ]
    ].set_index("stock_code").join(recent).join(previous).reset_index()
)

numeric_columns = [
    "recent_units",
    "recent_revenue",
    "recent_active_days",
    "previous_units",
    "previous_revenue",
    "previous_active_days",
]

trend[numeric_columns] = trend[numeric_columns].fillna(0)

# Both windows are 90 days long, so total-unit comparison is fair.
trend["recent_avg_daily_units"] = trend["recent_units"] / 90
trend["previous_avg_daily_units"] = trend["previous_units"] / 90

# Percentage change is undefined when previous demand is zero.
trend["demand_change_pct"] = np.where(trend["previous_units"] > 0,
    ((trend["recent_units"] - trend["previous_units"])/ trend["previous_units"]) * 100,np.nan,
)

# Avoid interpreting percentage changes on negligible volumes.
MIN_PREVIOUS_UNITS = 10

trend["trend_status"] = np.select(
    [
        trend["previous_units"] < MIN_PREVIOUS_UNITS,
        trend["recent_units"] == 0,
        trend["demand_change_pct"] >= 20,
        trend["demand_change_pct"] <= -20,
    ],
    [
        "Insufficient previous demand",
        "No recent recorded demand",
        "Increasing",
        "Declining",
    ],
    default="Relatively stable",
)


# 2. Summarize the overall trend

print("\n" + "=" * 70)
print("2. TREND STATUS DISTRIBUTION")
print("=" * 70)

print(trend["trend_status"].value_counts().to_string())

print("\n" + "=" * 70)
print("3. RECENT VERSUS PREVIOUS SALES")
print("=" * 70)

print("Recent-window units:", int(trend["recent_units"].sum()))
print("Previous-window units:", int(trend["previous_units"].sum()))

print("Recent-window revenue:", round(trend["recent_revenue"].sum(), 2))
print("Previous-window revenue:", round(trend["previous_revenue"].sum(), 2))


# 3. Display candidate increasing and declining products.


trend_columns = [
    "stock_code",
    "description",
    "movement_class",
    "history_quality",
    "previous_units",
    "recent_units",
    "demand_change_pct",
    "trend_status",
]

print("\n" + "=" * 70)
print("4. PRODUCTS WITH DECLINING RECORDED DEMAND")
print("=" * 70)

declining = trend.loc[(trend["trend_status"] == "Declining")& (trend["previous_units"] >= MIN_PREVIOUS_UNITS)
].sort_values("demand_change_pct")

print(declining[trend_columns].head(15).round(2).to_string(index=False))

print("\n" + "=" * 70)
print("5. PRODUCTS WITH INCREASING RECORDED DEMAND")
print("=" * 70)

increasing = trend.loc[trend["trend_status"] == "Increasing"].sort_values("demand_change_pct", ascending=False)

print(increasing[trend_columns].head(15).round(2).to_string(index=False))


# 4. Validation

print("\n" + "=" * 70)
print("6. TREND DATA VALIDATION")
print("=" * 70)

print("Products in trend table:", len(trend))
print("Duplicate stock codes:", trend["stock_code"].duplicated().sum())
print("Missing trend statuses:", trend["trend_status"].isna().sum())

print("\nTrend analysis complete. No files saved.")


1. COMPARISON WINDOWS
Dataset end date: 2011-12-09
Recent window: 2011-09-11 to 2011-12-09
Previous window: 2011-06-13 to 2011-09-10

2. TREND STATUS DISTRIBUTION
trend_status
Insufficient previous demand    2421
Increasing                      1373
Declining                        595
Relatively stable                415
No recent recorded demand        113

3. RECENT VERSUS PREVIOUS SALES
Recent-window units: 2109425
Previous-window units: 1210923
Recent-window revenue: 4096043.24
Previous-window revenue: 2174254.0

4. PRODUCTS WITH DECLINING RECORDED DEMAND
stock_code                        description movement_class             history_quality  previous_units  recent_units  demand_change_pct trend_status
     23134    LARGE ZINC HEART WALL ORGANISER  Medium-moving Sufficient observed history           812.0           1.0             -99.88    Declining
    84251B       GREETING CARD, STICKY GORDON    Slow-moving Sufficient observed history           108.0           1.0             

In [20]:
# RetailPulse — Step 15: Seasonality and Trend Diagnostics

# 1. Monthly sales across the full historical dataset
sales_diag = sales.copy()
sales_diag["sales_date"] = sales_diag["invoice_date"].dt.normalize()
sales_diag["month"] = sales_diag["sales_date"].dt.to_period("M")

monthly_sales = (sales_diag.groupby("month", observed=True).agg(
        transaction_rows=("invoice", "size"),
        unique_invoices=("invoice", "nunique"),
        active_products=("stock_code", "nunique"),
        units_sold=("quantity", "sum"),
        revenue=("revenue", "sum"),
    ).reset_index()
)

print("=" * 70)
print("1. MONTHLY SALES HISTORY")
print("=" * 70)
print(monthly_sales.tail(18).to_string(index=False))

# 2. Compare recent 90 days with the same dates one year earlier.
# Both windows are inclusive and contain 90 calendar days.
yoy_recent_start = recent_start
yoy_recent_end = data_end

yoy_previous_start = yoy_recent_start - pd.DateOffset(years=1)
yoy_previous_end = yoy_recent_end - pd.DateOffset(years=1)

def summarize_window(df, start, end):
    part = df.loc[df["sales_date"].between(start, end)]

    return {
        "start": start.date(),
        "end": end.date(),
        "days": (end - start).days + 1,
        "units": int(part["daily_units_sold"].sum()),
        "revenue": float(part["daily_revenue"].sum()),
        "product_days": len(part),
        "active_products": part["stock_code"].nunique(),
        "active_calendar_days": part["sales_date"].nunique(),
    }

recent_summary = summarize_window(daily, yoy_recent_start, yoy_recent_end)
yoy_summary = summarize_window(daily, yoy_previous_start, yoy_previous_end)

comparison = pd.DataFrame([
    {"period": "Recent period", **recent_summary},
    {"period": "Same period previous year", **yoy_summary},
])

print("\n" + "=" * 70)
print("2. SAME-SEASON YEAR-OVER-YEAR COMPARISON")
print("=" * 70)
print(comparison.to_string(index=False))

if yoy_summary["units"] > 0:
    units_change = ((recent_summary["units"] - yoy_summary["units"])/ yoy_summary["units"] * 100)
    print(f"\nYear-over-year unit change: {units_change:.2f}%")

if yoy_summary["revenue"] > 0:
    revenue_change = ((recent_summary["revenue"] - yoy_summary["revenue"])/ yoy_summary["revenue"] * 100)
    print(f"Year-over-year revenue change: {revenue_change:.2f}%")

# 3. Compare product-level demand with the same season last year.
recent_product = (
    daily.loc[daily["sales_date"].between(yoy_recent_start, yoy_recent_end)]
    .groupby("stock_code", observed=True)["daily_units_sold"].sum().rename("recent_units")
)

previous_product = (
    daily.loc[daily["sales_date"].between(yoy_previous_start, yoy_previous_end)]
    .groupby("stock_code", observed=True)["daily_units_sold"]
    .sum().rename("same_season_previous_units")
)

yoy_products = (
    profiles[["stock_code", "description", "movement_class", "history_quality"]]
    .set_index("stock_code").join(recent_product).join(previous_product)
    .fillna({"recent_units": 0,"same_season_previous_units": 0,}).reset_index()
)

# Require a minimum prior-year volume before calculating percentages.
MIN_BASELINE_UNITS = 10

yoy_products["yoy_change_pct"] = np.where(
    yoy_products["same_season_previous_units"] >= MIN_BASELINE_UNITS,
    ((yoy_products["recent_units"]- yoy_products["same_season_previous_units"])/ yoy_products["same_season_previous_units"]) * 100,np.nan,
)

yoy_products["yoy_change_units"] = (yoy_products["recent_units"]- yoy_products["same_season_previous_units"])

print("\n" + "=" * 70)
print("3. PRODUCT-LEVEL YEAR-OVER-YEAR COVERAGE")
print("=" * 70)

print("Products:", len(yoy_products))
print("Products with a sufficient prior-year baseline:",int(yoy_products["yoy_change_pct"].notna().sum()))
print("Products with zero recent units:",int((yoy_products["recent_units"] == 0).sum()))

print("\nLargest positive same-season changes by absolute units:")
print(yoy_products.sort_values("yoy_change_units", ascending=False)[
        [
            "stock_code", "description", "movement_class",
            "same_season_previous_units", "recent_units",
            "yoy_change_units", "yoy_change_pct",
        ]
    ].head(15).round(2).to_string(index=False)
)

print("\nLargest negative same-season changes by absolute units:")
print(yoy_products.sort_values("yoy_change_units")[
        [
            "stock_code", "description", "movement_class",
            "same_season_previous_units", "recent_units",
            "yoy_change_units", "yoy_change_pct",
        ]
    ].head(15).round(2).to_string(index=False)
)

print("\nDiagnostics complete. No files saved or modified.")


1. MONTHLY SALES HISTORY
  month  transaction_rows  unique_invoices  active_products  units_sold     revenue
2010-07             32112             1529             2612      337895  648810.270
2010-08             32109             1425             2688      472380  695251.910
2010-09             40630             1839             2719      583864  921696.991
2010-10             56953             2301             2842      619996 1161902.220
2010-11             75035             2747             3033      725161 1464293.142
2010-12             40991             1559             2788      358019  821452.730
2011-01             34060             1086             2573      387099  689811.610
2011-02             26882             1100             2398      282934  522545.560
2011-03             35497             1454             2503      376599  716215.260
2011-04             28882             1246             2459      307953  536968.491
2011-05             35917             1681         

In [21]:
# RetailPulse — Step 17: Finalize Product Demand Trends

DATA_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# 1. Combine short-term and same-season comparisons

trend_final = trend.copy()

yoy_columns = [
    "stock_code",
    "same_season_previous_units",
    "yoy_change_units",
    "yoy_change_pct",
]

trend_final = trend_final.merge(yoy_products[yoy_columns],on="stock_code",how="left",validate="one_to_one",)

# Explicitly classify year-over-year movement.
# Percentage changes require a meaningful prior-year baseline.

MIN_BASELINE_UNITS = 10

trend_final["yoy_trend_status"] = np.select(
    [
        trend_final["same_season_previous_units"] < MIN_BASELINE_UNITS,
        trend_final["recent_units"] == 0,
        trend_final["yoy_change_pct"] >= 20,
        trend_final["yoy_change_pct"] <= -20,
    ],
    [
        "Insufficient comparable baseline",
        "No recent recorded demand",
        "Increasing vs same season",
        "Declining vs same season",
    ],
    default="Relatively stable vs same season",
)


# 2. Create a conservative historical priority indicator
# This is a screening indicator, NOT a stockout prediction.
# It combines movement class with recent sales decline.
# Products without sufficient history remain separately flagged.

trend_final["historical_demand_attention"] = np.select(
    [
        trend_final["history_quality"] == "Limited observed history",
        ((trend_final["trend_status"] == "Declining")& (trend_final["yoy_trend_status"] == "Declining vs same season")),
        ((trend_final["trend_status"] == "Declining")| (trend_final["yoy_trend_status"] == "Declining vs same season")),
    ],
    [
        "Review limited history",
        "Consistent historical decline",
        "Review demand decline",
    ],
    default="Routine historical review",
)

# 3. Validate product-level results

print("=" * 70)
print("1. FINAL TREND DATASET")
print("=" * 70)

print("Shape:", trend_final.shape)
print("Unique products:", trend_final["stock_code"].nunique())
print("Duplicate stock codes:", trend_final["stock_code"].duplicated().sum())
print("Missing short-term statuses:", trend_final["trend_status"].isna().sum())
print("Missing year-over-year statuses:", trend_final["yoy_trend_status"].isna().sum())

print("\nShort-term trend statuses:")
print(trend_final["trend_status"].value_counts().to_string())

print("\nSame-season year-over-year statuses:")
print(trend_final["yoy_trend_status"].value_counts().to_string())

print("\nHistorical attention categories:")
print(trend_final["historical_demand_attention"].value_counts().to_string())


# 4. Summarize trends by movement class

trend_summary = (
    trend_final.groupby("movement_class", observed=True).agg(
        product_count=("stock_code", "nunique"),
        recent_units=("recent_units", "sum"),
        previous_units=("previous_units", "sum"),
        same_season_previous_units=("same_season_previous_units", "sum"),
        recent_revenue=("recent_revenue", "sum"),
        previous_revenue=("previous_revenue", "sum"),
        products_with_limited_history=("history_quality",lambda x: (x == "Limited observed history").sum()),
        products_with_recent_zero_sales=("recent_units",lambda x: (x == 0).sum()),
    ).reset_index()
)

print("\n" + "=" * 70)
print("2. TREND SUMMARY BY MOVEMENT CLASS")
print("=" * 70)

print(trend_summary.round(2).to_string(index=False))


# 5. Save new files without overwriting existing outputs

trend_path = DATA_DIR / "product_demand_trends.csv"
summary_path = OUTPUT_DIR / "inventory_trend_summary.csv"

for path in [trend_path, summary_path]:
    if path.exists():
        raise FileExistsError(f"{path} already exists. Inspect it before overwriting.")

trend_final.to_csv(trend_path, index=False)
trend_summary.to_csv(summary_path, index=False)

print("\n" + "=" * 70)
print("3. SAVED OUTPUTS")
print("=" * 70)

print(trend_path)
print(summary_path)


# 6. Reload and validate saved files

saved_trends = pd.read_csv(trend_path)
saved_summary = pd.read_csv(summary_path)

assert len(saved_trends) == len(profiles)
assert saved_trends["stock_code"].is_unique
assert saved_trends["trend_status"].notna().all()
assert saved_trends["yoy_trend_status"].notna().all()
assert len(saved_summary) == 3

print("\nSaved trend dataset:", saved_trends.shape)
print("Saved summary:", saved_summary.shape)
print("All validation checks passed.")
print("\nNo existing files were overwritten.")


1. FINAL TREND DATASET
Shape: (4917, 20)
Unique products: 4917
Duplicate stock codes: 0
Missing short-term statuses: 0
Missing year-over-year statuses: 0

Short-term trend statuses:
trend_status
Insufficient previous demand    2421
Increasing                      1373
Declining                        595
Relatively stable                415
No recent recorded demand        113

Same-season year-over-year statuses:
yoy_trend_status
Insufficient comparable baseline    2184
Declining vs same season            1182
Increasing vs same season            591
No recent recorded demand            566
Relatively stable vs same season     394

Historical attention categories:
historical_demand_attention
Routine historical review        2750
Review demand decline            1075
Review limited history            741
Consistent historical decline     351

2. TREND SUMMARY BY MOVEMENT CLASS
movement_class  product_count  recent_units  previous_units  same_season_previous_units  recent_revenue  previ

In [23]:
# Build Inventory Priority Dataset

DATA_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs"


# 1. Load the validated saved datasets

profiles = pd.read_parquet(DATA_DIR / "product_demand_profiles.parquet")

trend_final = pd.read_csv(DATA_DIR / "product_demand_trends.csv",dtype={"stock_code": "string"})

print("=" * 70)
print("1. INPUT DATASETS")
print("=" * 70)

print("Demand profiles:", profiles.shape)
print("Demand trends:", trend_final.shape)

assert profiles["stock_code"].is_unique
assert trend_final["stock_code"].is_unique


# 2. Select trend fields and merge one-to-one by product

trend_columns = [
    "stock_code",
    "recent_units",
    "recent_revenue",
    "recent_active_days",
    "previous_units",
    "previous_revenue",
    "previous_active_days",
    "recent_avg_daily_units",
    "previous_avg_daily_units",
    "demand_change_pct",
    "trend_status",
    "same_season_previous_units",
    "yoy_change_units",
    "yoy_change_pct",
    "yoy_trend_status",
    "historical_demand_attention",
]

missing_trend_columns = [
    col for col in trend_columns
    if col not in trend_final.columns
]

if missing_trend_columns:
    raise KeyError(f"Missing expected trend columns: {missing_trend_columns}")

priority = profiles.merge(
    trend_final[trend_columns],
    on="stock_code",
    how="left",
    validate="one_to_one",
    indicator=True,
)

print("\nMerge results:")
print(priority["_merge"].value_counts().to_string())

unmatched = priority.loc[
    priority["_merge"] != "both", "stock_code"
]

if not unmatched.empty:
    raise ValueError(f"{len(unmatched)} products could not be matched. " "Inspect stock codes before proceeding.")

priority = priority.drop(columns="_merge")


# 3. Validate required analytical fields

required_fields = [
    "stock_code",
    "description",
    "units_sold",
    "revenue",
    "movement_class",
    "history_quality",
    "active_selling_days",
    "cv_demand",
    "demand_occurrence_rate",
    "recent_units",
    "previous_units",
    "trend_status",
    "yoy_trend_status",
]

missing_fields = [
    col for col in required_fields
    if col not in priority.columns
]

if missing_fields:
    raise KeyError(f"Missing required fields: {missing_fields}")

# Missing sales in a window mean no recorded sales for that product.
# These are not evidence of stockouts.
for col in ["recent_units", "previous_units", "recent_revenue",
            "previous_revenue", "recent_active_days",
            "previous_active_days"]:
    priority[col] = priority[col].fillna(0)


# 4. Create independent review flags


priority["fast_moving_flag"] = (priority["movement_class"] == "Fast-moving")

priority["limited_history_flag"] = (priority["history_quality"] == "Limited observed history")

priority["no_recent_sales_flag"] = (priority["recent_units"] == 0)

priority["short_term_growth_flag"] = (priority["trend_status"] == "Increasing")

priority["same_season_growth_flag"] = (priority["yoy_trend_status"] == "Increasing vs same season")

priority["sustained_decline_flag"] = ((priority["trend_status"] == "Declining")& (
        priority["yoy_trend_status"]
        == "Declining vs same season"
    )
)

priority["growth_signal_flag"] = (priority["short_term_growth_flag"]| priority["same_season_growth_flag"])


# 5. Flag unusually variable demand
# Calculate the 75th percentile only among products with
# sufficient observed history and at least five active days.

reliable_history = (~priority["limited_history_flag"] & (priority["active_selling_days"] >= 5))

reliable_cv = priority.loc[reliable_history, "cv_demand"].replace([np.inf, -np.inf], np.nan).dropna()

if reliable_cv.empty:
    raise ValueError("Cannot calculate the variability threshold: " "no eligible products have valid CV values.")

CV_THRESHOLD = reliable_cv.quantile(0.75)

priority["high_variability_flag"] = (reliable_history & (priority["cv_demand"] >= CV_THRESHOLD))

print("\n" + "=" * 70)
print("2. VARIABILITY SCREEN")
print("=" * 70)

print("Eligible products:", len(reliable_cv))
print("75th percentile CV threshold:", round(CV_THRESHOLD, 4))
print("Products flagged for high variability:",int(priority["high_variability_flag"].sum()))


# 6. Assign an explainable primary review category
# Priority order prevents weak historical evidence from being
# presented as a confident operational recommendation.

conditions = [
    priority["limited_history_flag"],
    priority["no_recent_sales_flag"],
    priority["sustained_decline_flag"],
    (priority["fast_moving_flag"]& priority["growth_signal_flag"]),
    priority["fast_moving_flag"],
    priority["high_variability_flag"],
]

categories = [
    "Limited history review",
    "No recent recorded demand",
    "Sustained decline review",
    "High demand / growth review",
    "Fast-moving review",
    "Volatility review",
]

priority["review_priority"] = np.select(conditions,categories,default="Routine review",)

# A broad flag for any non-routine primary category.
priority["requires_historical_review"] = (priority["review_priority"] != "Routine review")


# 7. Create a concise inventory-priority dataset

priority_columns = [
    "stock_code",
    "description",
    "movement_class",
    "history_quality",
    "units_sold",
    "revenue",
    "active_selling_days",
    "mean_daily_demand",
    "std_daily_demand",
    "cv_demand",
    "demand_occurrence_rate",
    "recent_units",
    "previous_units",
    "demand_change_pct",
    "same_season_previous_units",
    "yoy_change_units",
    "yoy_change_pct",
    "trend_status",
    "yoy_trend_status",
    "historical_demand_attention",
    "fast_moving_flag",
    "limited_history_flag",
    "no_recent_sales_flag",
    "growth_signal_flag",
    "sustained_decline_flag",
    "high_variability_flag",
    "review_priority",
    "requires_historical_review",
]

priority_dataset = priority[priority_columns].copy()

print("\n" + "=" * 70)
print("3. PRIMARY REVIEW PRIORITIES")
print("=" * 70)

priority_counts = (priority_dataset["review_priority"].value_counts().rename_axis("review_priority").reset_index(name="product_count"))

priority_counts["product_share"] = (priority_counts["product_count"]/ len(priority_dataset))

print(priority_counts.round(4).to_string(index=False))

print("\nIndependent review flags:")
for col in [
    "fast_moving_flag",
    "limited_history_flag",
    "no_recent_sales_flag",
    "growth_signal_flag",
    "sustained_decline_flag",
    "high_variability_flag",
]:
    print(f"{col}: {int(priority_dataset[col].sum())}")


# 8. Inspect examples from each primary review category

print("\n" + "=" * 70)
print("4. PRODUCTS REQUIRING REVIEW")
print("=" * 70)

example_columns = [
    "stock_code",
    "description",
    "movement_class",
    "units_sold",
    "recent_units",
    "previous_units",
    "demand_change_pct",
    "yoy_trend_status",
    "review_priority",
]

for category in categories:
    examples = priority_dataset.loc[priority_dataset["review_priority"] == category]

    print(f"\n{category}: {len(examples)} products")

    print(examples.sort_values("units_sold", ascending=False)[example_columns].head(5).round(2).to_string(index=False))


# 9. Final validation

print("\n" + "=" * 70)
print("5. PRIORITY DATASET VALIDATION")
print("=" * 70)

print("Input profile products:", len(profiles))
print("Priority dataset products:", len(priority_dataset))
print("Duplicate stock codes:", priority_dataset["stock_code"].duplicated().sum())
print("Missing priority labels:", priority_dataset["review_priority"].isna().sum())
print("Products accounted for by primary categories:",int(priority_dataset["review_priority"].value_counts().sum()))

assert len(priority_dataset) == len(profiles)
assert priority_dataset["stock_code"].is_unique
assert priority_dataset["review_priority"].notna().all()
assert priority_dataset["review_priority"].value_counts().sum() == len(profiles)

assert np.isclose(priority_dataset["units_sold"].sum(),profiles["units_sold"].sum())

assert np.isclose(priority_dataset["revenue"].sum(),profiles["revenue"].sum())

print("\nAll priority dataset validation checks passed.")
print("No files saved or overwritten.")


1. INPUT DATASETS
Demand profiles: (4917, 19)
Demand trends: (4917, 20)

Merge results:
_merge
both          4917
left_only        0
right_only       0

2. VARIABILITY SCREEN
Eligible products: 4176
75th percentile CV threshold: 6.7055
Products flagged for high variability: 1044

3. PRIMARY REVIEW PRIORITIES
            review_priority  product_count  product_share
  No recent recorded demand           1187         0.2414
             Routine review           1089         0.2215
     Limited history review            741         0.1507
High demand / growth review            696         0.1415
          Volatility review            554         0.1127
   Sustained decline review            351         0.0714
         Fast-moving review            299         0.0608

Independent review flags:
fast_moving_flag: 1639
limited_history_flag: 741
no_recent_sales_flag: 1683
growth_signal_flag: 1583
sustained_decline_flag: 351
high_variability_flag: 1044

4. PRODUCTS REQUIRING REVIEW

Limited his

In [25]:
# Step 19 — Save inventory priority outputs safely


# Keep the existing inventory_priority_summary.csv unchanged.
dataset_path = DATA_DIR / "inventory_priority_dataset.csv"
summary_path = OUTPUT_DIR / "inventory_review_priority_summary.csv"

# Only stop if the NEW output filenames already exist.
existing_paths = [
    path for path in [dataset_path, summary_path]
    if path.exists()
]

if existing_paths:
    raise FileExistsError(
        "These output files already exist. Inspect them first:\n"
        + "\n".join(str(path) for path in existing_paths)
    )

# Save the product-level dataset and new category summary.
priority_dataset.to_csv(dataset_path, index=False)
priority_summary.to_csv(summary_path, index=False)

print("Saved product-level dataset:", dataset_path)
print("Saved review-priority summary:", summary_path)

# Reload and validate
saved_priority = pd.read_csv(dataset_path,dtype={"stock_code": "string"},)
saved_summary = pd.read_csv(summary_path)

assert len(saved_priority) == len(priority_dataset)
assert saved_priority["stock_code"].is_unique
assert saved_priority["review_priority"].notna().all()

assert np.isclose(saved_priority["units_sold"].sum(),priority_dataset["units_sold"].sum(),)
assert np.isclose(saved_priority["revenue"].sum(),priority_dataset["revenue"].sum(),)
assert int(saved_summary["product_count"].sum()) == len(priority_dataset)

print("\nSaved dataset shape:", saved_priority.shape)
print("Saved summary shape:", saved_summary.shape)
print("Duplicate product codes:", saved_priority["stock_code"].duplicated().sum())
print("All validation checks passed.")
print("Existing project files were not overwritten.")


Saved product-level dataset: e:\RetailPulse\data\processed\inventory_priority_dataset.csv
Saved review-priority summary: e:\RetailPulse\outputs\inventory_review_priority_summary.csv

Saved dataset shape: (4917, 28)
Saved summary shape: (7, 11)
Duplicate product codes: 0
All validation checks passed.
Existing project files were not overwritten.


In [26]:
DATA_DIR = PROJECT_ROOT / "data" / "processed"

# Inventory-related files to look for
candidate_files = [
    "clean_sales.parquet",
    "clean_transactions.parquet",
    "clean_returns.parquet",
    "clean_cancelled.parquet",
    "daily_demand.parquet",
    "product_demand_profiles.parquet",
    "product_demand_trends.csv",
    "inventory_priority_dataset.csv",
]

# Search for operational inventory-related files by filename
print("Inventory data readiness inspection")
print(f"Processed directory: {DATA_DIR}")

for path in sorted(DATA_DIR.iterdir()):
    if path.is_file() and any(
        keyword in path.name.lower()
        for keyword in [
            "stock", "inventory", "supplier", "purchase",
            "order", "lead", "warehouse", "product", "sales",
            "transaction", "return", "cancel", "demand"
        ]
    ):
        print(path.name)

print("\nCandidate dataset schemas")

for filename in candidate_files:
    path = DATA_DIR / filename

    if not path.exists():
        print(f"\n{filename}: NOT FOUND")
        continue

    if path.suffix.lower() == ".parquet":
        df = pd.read_parquet(path)
    elif path.suffix.lower() == ".csv":
        df = pd.read_csv(path, nrows=5)
    else:
        continue

    print(f"\n{filename}")
    print("Columns:", df.columns.tolist())

# Check whether the main sales dataset contains operational fields
sales = pd.read_parquet(DATA_DIR / "clean_sales.parquet")

operational_keywords = [
    "stock_on_hand", "on_hand", "inventory_level",
    "stock_quantity", "reorder_point", "safety_stock",
    "lead_time", "supplier", "purchase_order",
    "warehouse_stock", "stockout", "unit_cost"
]

found_fields = [
    column for column in sales.columns
    if any(keyword in column.lower() for keyword in operational_keywords)
]

print("\nOperational fields in clean_sales.parquet:")
print(found_fields if found_fields else "None detected")

print("\nInspection complete. No files modified.")


Inventory data readiness inspection
Processed directory: e:\RetailPulse\data\processed
clean_cancelled.parquet
clean_returns.parquet
clean_sales.parquet
clean_transactions.parquet
daily_demand.parquet
final_demand_forecast.csv
inventory_priority_dataset.csv
product_demand_profiles.parquet
product_demand_trends.csv
product_movement_analysis.csv

Candidate dataset schemas

clean_sales.parquet
Columns: ['invoice', 'stock_code', 'description', 'quantity', 'invoice_date', 'unit_price', 'customer_id', 'country', 'is_cancelled', 'revenue', 'description_clean']

clean_transactions.parquet
Columns: ['invoice', 'stock_code', 'description', 'quantity', 'invoice_date', 'unit_price', 'customer_id', 'country', 'is_cancelled', 'revenue', 'description_clean']

clean_returns.parquet
Columns: ['invoice', 'stock_code', 'description', 'quantity', 'invoice_date', 'unit_price', 'customer_id', 'country', 'is_cancelled', 'revenue', 'description_clean']

clean_cancelled.parquet
Columns: ['invoice', 'stock_code

In [27]:
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

readiness_records = [
    {
        "capability": "Historical product movement",
        "status": "Available",
        "required_data": "Product identifiers and historical sales",
        "current_evidence": "Validated product movement analysis",
        "limitation": "Describes historical sales, not current stock",
    },
    {
        "capability": "Demand variability",
        "status": "Available",
        "required_data": "Daily product demand",
        "current_evidence": "Product demand profiles and variability metrics",
        "limitation": "Historical variability may include seasonal and bulk-order effects",
    },
    {
        "capability": "Historical demand trends",
        "status": "Available",
        "required_data": "Historical sales with comparable periods",
        "current_evidence": "Short-term and same-season trend comparisons",
        "limitation": "Historical trends do not guarantee future demand",
    },
    {
        "capability": "Current stock on hand",
        "status": "Unavailable",
        "required_data": "Current product-level stock balances",
        "current_evidence": "No stock balance fields found in inspected datasets",
        "limitation": "Cannot determine actual available inventory",
    },
    {
        "capability": "Supplier lead-time analysis",
        "status": "Unavailable",
        "required_data": "Supplier and product lead-time records",
        "current_evidence": "No lead-time or supplier fields found",
        "limitation": "Cannot calculate supplier-specific replenishment timing",
    },
    {
        "capability": "Purchase-order tracking",
        "status": "Unavailable",
        "required_data": "Purchase orders, quantities, order dates, and receipt dates",
        "current_evidence": "No purchase-order dataset identified",
        "limitation": "Cannot calculate incoming stock or actual replenishment performance",
    },
    {
        "capability": "Actual stockout detection",
        "status": "Unavailable",
        "required_data": "Stock availability history or explicit stockout records",
        "current_evidence": "No stockout or availability fields found",
        "limitation": "Zero sales does not prove a stockout",
    },
    {
        "capability": "Safety stock and reorder points",
        "status": "Blocked by missing inputs",
        "required_data": "Demand estimates, service-level target, and lead-time assumptions",
        "current_evidence": "Historical demand metrics available; operational inputs missing",
        "limitation": "Any numerical result would require explicit assumptions",
    },
    {
        "capability": "Replenishment quantity recommendations",
        "status": "Blocked by missing inputs",
        "required_data": "Current stock, incoming orders, demand, and replenishment constraints",
        "current_evidence": "Historical demand and review priorities available",
        "limitation": "Cannot calculate actionable order quantities",
    },
]

readiness = pd.DataFrame(readiness_records)

report_path = OUTPUT_DIR / "inventory_data_readiness.csv"

if report_path.exists():
    raise FileExistsError(f"{report_path} already exists. Inspect it before overwriting.")

readiness.to_csv(report_path, index=False)

print("Inventory readiness report saved:")
print(report_path)

print("\nReadiness summary:")
print(readiness[["capability", "status"]].to_string(index=False))

print("\nTotal capabilities assessed:", len(readiness))
print("Available:", (readiness["status"] == "Available").sum())
print("Unavailable:", (readiness["status"] == "Unavailable").sum())
print("Blocked by missing inputs:", (readiness["status"] == "Blocked by missing inputs").sum())

print("\nNo existing files were modified.")


Inventory readiness report saved:
e:\RetailPulse\outputs\inventory_data_readiness.csv

Readiness summary:
                            capability                    status
           Historical product movement                 Available
                    Demand variability                 Available
              Historical demand trends                 Available
                 Current stock on hand               Unavailable
           Supplier lead-time analysis               Unavailable
               Purchase-order tracking               Unavailable
             Actual stockout detection               Unavailable
       Safety stock and reorder points Blocked by missing inputs
Replenishment quantity recommendations Blocked by missing inputs

Total capabilities assessed: 9
Available: 3
Unavailable: 4
Blocked by missing inputs: 2

No existing files were modified.


In [28]:
project_root = Path.cwd()

# Locate the project root if the notebook is running from notebooks/
if project_root.name == "notebooks":
    project_root = project_root.parent

files_to_check = {
    "priority_dataset": project_root / "data" / "processed" / "inventory_priority_dataset.csv",
    "demand_profiles": project_root / "data" / "processed" / "product_demand_profiles.parquet",
    "demand_trends": project_root / "data" / "processed" / "product_demand_trends.csv",
    "priority_summary": project_root / "outputs" / "inventory_review_priority_summary.csv",
    "trend_summary": project_root / "outputs" / "inventory_trend_summary.csv",
    "readiness_report": project_root / "outputs" / "inventory_data_readiness.csv",
}

for name, path in files_to_check.items():
    print(f"\n{name}: {path}")
    if not path.exists():
        print("  MISSING")
        continue

    if path.suffix == ".parquet":
        df_check = pd.read_parquet(path)
    else:
        df_check = pd.read_csv(path)

    print(f"  Shape: {df_check.shape}")
    print(f"  Columns: {df_check.columns.tolist()}")



priority_dataset: e:\RetailPulse\data\processed\inventory_priority_dataset.csv
  Shape: (4917, 28)
  Columns: ['stock_code', 'description', 'movement_class', 'history_quality', 'units_sold', 'revenue', 'active_selling_days', 'mean_daily_demand', 'std_daily_demand', 'cv_demand', 'demand_occurrence_rate', 'recent_units', 'previous_units', 'demand_change_pct', 'same_season_previous_units', 'yoy_change_units', 'yoy_change_pct', 'trend_status', 'yoy_trend_status', 'historical_demand_attention', 'fast_moving_flag', 'limited_history_flag', 'no_recent_sales_flag', 'growth_signal_flag', 'sustained_decline_flag', 'high_variability_flag', 'review_priority', 'requires_historical_review']

demand_profiles: e:\RetailPulse\data\processed\product_demand_profiles.parquet
  Shape: (4917, 19)
  Columns: ['stock_code', 'first_sale', 'last_sale', 'observed_lifespan_days', 'active_selling_days', 'zero_sales_days', 'units_sold', 'mean_daily_demand', 'std_daily_demand', 'cv_demand', 'demand_occurrence_rate',

In [30]:
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

print("Project root:", project_root)

print("\nTop-level files and folders:")
for item in sorted(project_root.iterdir()):
    if item.name not in {".git", ".venv", "venv", "__pycache__"}:
        print(f"{'[DIR] ' if item.is_dir() else '[FILE]'} {item.name}")

print("\nPython files that may be Streamlit entry points:")
for path in project_root.rglob("*.py"):
    if any(part in {".git", ".venv", "venv", "__pycache__"} for part in path.parts):
        continue
    print(path.relative_to(project_root))


Project root: e:\RetailPulse

Top-level files and folders:
[FILE] .gitignore
[DIR]  .streamlit
[DIR]  .venv311
[DIR]  .vscode
[FILE] app.py
[FILE] app_backup.py
[DIR]  dashboard
[DIR]  data
[FILE] Dockerfile
[DIR]  monitoring
[DIR]  notebooks
[DIR]  outputs
[DIR]  pages
[FILE] pyproject.toml
[FILE] README.md
[FILE] requirements.txt
[DIR]  src
[FILE] test.py

Python files that may be Streamlit entry points:
app.py
app_backup.py
test.py
.venv311\Lib\site-packages\ipykernel_launcher.py
.venv311\Lib\site-packages\ipython_pygments_lexers.py
.venv311\Lib\site-packages\jupyter.py
.venv311\Lib\site-packages\nest_asyncio2.py
.venv311\Lib\site-packages\pylab.py
.venv311\Lib\site-packages\six.py
.venv311\Lib\site-packages\threadpoolctl.py
.venv311\Lib\site-packages\typing_extensions.py
.venv311\Lib\site-packages\absl\app.py
.venv311\Lib\site-packages\absl\command_name.py
.venv311\Lib\site-packages\absl\__init__.py
.venv311\Lib\site-packages\absl\flags\argparse_flags.py
.venv311\Lib\site-packages\

In [1]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

sales_path = project_root / "data" / "processed" / "clean_sales.parquet"
output_path = (
    project_root
    / "data"
    / "processed"
    / "product_daily_demand.parquet"
)

if not sales_path.exists():
    raise FileNotFoundError(f"Sales dataset not found: {sales_path}")

if output_path.exists():
    raise FileExistsError(
        f"{output_path.name} already exists. "
        "Inspect it before deciding whether to reuse or replace it."
    )

sales = pd.read_parquet(sales_path)

required_columns = {
    "stock_code",
    "invoice_date",
    "quantity",
    "unit_price",
    "is_cancelled",
}

missing_columns = required_columns - set(sales.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

sales["invoice_date"] = pd.to_datetime(
    sales["invoice_date"], errors="coerce"
)
sales["quantity"] = pd.to_numeric(sales["quantity"], errors="coerce")
sales["unit_price"] = pd.to_numeric(sales["unit_price"], errors="coerce")

valid_sales = sales.loc[
    sales["stock_code"].notna()
    & sales["invoice_date"].notna()
    & sales["quantity"].gt(0)
    & sales["unit_price"].gt(0)
    & ~sales["is_cancelled"].fillna(True).astype(bool)
].copy()

valid_sales["date"] = valid_sales["invoice_date"].dt.normalize()

product_daily_demand = (
    valid_sales.groupby(
        ["stock_code", "date"],
        as_index=False,
        observed=True,
    )
    .agg(
        daily_units=("quantity", "sum"),
        daily_revenue=("quantity", lambda s: 0),
    )
)

# Calculate revenue from valid transactions separately.
daily_revenue = (
    valid_sales.assign(
        transaction_revenue=valid_sales["quantity"]
        * valid_sales["unit_price"]
    )
    .groupby(["stock_code", "date"], as_index=False)
    .agg(daily_revenue=("transaction_revenue", "sum"))
)

product_daily_demand = product_daily_demand.drop(
    columns=["daily_revenue"]
).merge(
    daily_revenue,
    on=["stock_code", "date"],
    how="left",
    validate="one_to_one",
)

product_daily_demand = product_daily_demand.sort_values(
    ["stock_code", "date"]
).reset_index(drop=True)

if product_daily_demand.duplicated(["stock_code", "date"]).any():
    raise ValueError("Duplicate product-date keys detected.")

if (product_daily_demand["daily_units"] <= 0).any():
    raise ValueError("Non-positive daily demand found in valid sales output.")

product_daily_demand.to_parquet(output_path, index=False)

print("Product-level daily demand saved:", output_path)
print("Shape:", product_daily_demand.shape)
print("Unique products:", product_daily_demand["stock_code"].nunique())
print("Unique product-date keys:",
      product_daily_demand[["stock_code", "date"]].drop_duplicates().shape[0])
print("Total units:", f"{product_daily_demand['daily_units'].sum():,.0f}")
print("Total revenue:", f"£{product_daily_demand['daily_revenue'].sum():,.2f}")
print("Date range:",
      product_daily_demand["date"].min(),
      "to",
      product_daily_demand["date"].max())
print("\nSample rows:")
print(product_daily_demand.head().to_string(index=False))


Product-level daily demand saved: e:\RetailPulse\data\processed\product_daily_demand.parquet
Shape: (532839, 4)
Unique products: 4917
Unique product-date keys: 532839
Total units: 11,205,148
Total revenue: £20,476,260.45
Date range: 2009-12-01 00:00:00 to 2011-12-09 00:00:00

Sample rows:
stock_code       date  daily_units  daily_revenue
     10002 2009-12-01           12          10.20
     10002 2009-12-03            6           5.10
     10002 2009-12-04           73          62.05
     10002 2009-12-06           49          41.65
     10002 2009-12-07            2           1.70


In [1]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

sales_path = project_root / "data" / "processed" / "clean_sales.parquet"

sales = pd.read_parquet(sales_path)

sales["stock_code"] = sales["stock_code"].astype(str).str.strip()
sales["invoice_date"] = pd.to_datetime(
    sales["invoice_date"], errors="coerce"
)

product_code = "23843"

product_sales = sales.loc[
    sales["stock_code"].eq(product_code)
    & sales["quantity"].gt(0)
    & sales["unit_price"].gt(0)
    & ~sales["is_cancelled"].fillna(True).astype(bool)
].copy()

product_sales["date"] = product_sales["invoice_date"].dt.normalize()

print("Product:", product_code)
print("Qualifying transaction rows:", len(product_sales))
print("Distinct invoices:", product_sales["invoice"].nunique())
print("Distinct sales dates:", product_sales["date"].nunique())
print("Total units:", f"{product_sales['quantity'].sum():,.0f}")
print("Total revenue:", f"£{product_sales['revenue'].sum():,.2f}")
print("Date range:", product_sales["date"].min(), "to", product_sales["date"].max())

print("\nUnits by date:")
print(
    product_sales.groupby("date", as_index=False)
    .agg(
        daily_units=("quantity", "sum"),
        daily_revenue=("revenue", "sum"),
        invoice_count=("invoice", "nunique"),
    )
    .sort_values("daily_units", ascending=False)
    .head(15)
    .to_string(index=False)
)


Product: 23843
Qualifying transaction rows: 1
Distinct invoices: 1
Distinct sales dates: 1
Total units: 80,995
Total revenue: £168,469.60
Date range: 2011-12-09 00:00:00 to 2011-12-09 00:00:00

Units by date:
      date  daily_units  daily_revenue  invoice_count
2011-12-09        80995       168469.6              1


In [2]:
invoice_details = product_sales[
    [
        "invoice",
        "invoice_date",
        "stock_code",
        "description",
        "quantity",
        "unit_price",
        "revenue",
    ]
]

print(invoice_details.to_string(index=False))


invoice        invoice_date stock_code                 description  quantity  unit_price  revenue
 581483 2011-12-09 09:15:00      23843 PAPER CRAFT , LITTLE BIRDIE     80995        2.08 168469.6


In [1]:

from pathlib import Path
import pandas as pd

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

files = {
    "Product profiles": (
        project_root / "data" / "processed"
        / "product_demand_profiles.parquet"
    ),
    "Demand trends": (
        project_root / "data" / "processed"
        / "product_demand_trends.csv"
    ),
    "Priority dataset": (
        project_root / "data" / "processed"
        / "inventory_priority_dataset.csv"
    ),
    "Daily demand": (
        project_root / "data" / "processed"
        / "product_daily_demand.parquet"
    ),
}

product_sets = {}

for name, path in files.items():
    print(f"\n{name}")
    print(f"File exists: {path.exists()}")

    if not path.exists():
        print(f"Missing file: {path}")
        continue

    if path.suffix == ".parquet":
        df = pd.read_parquet(path)
    else:
        df = pd.read_csv(path, dtype={"stock_code": "string"})

    if "stock_code" not in df.columns:
        print("ERROR: stock_code column is missing.")
        continue

    codes = df["stock_code"].astype("string").str.strip()
    product_sets[name] = set(codes.dropna())

    print("Rows:", f"{len(df):,}")
    print("Unique non-null stock codes:", f"{codes.nunique():,}")
    print("Missing stock codes:", f"{codes.isna().sum():,}")
    print("Duplicate stock-code rows:", f"{codes.duplicated().sum():,}")

if product_sets:
    print("\nProduct count comparison")
    for name, codes in product_sets.items():
        print(f"{name}: {len(codes):,} unique products")

    reference_name = "Product profiles"

    if reference_name in product_sets:
        reference_codes = product_sets[reference_name]

        print("\nDifferences compared with product profiles")
        for name, codes in product_sets.items():
            missing = reference_codes - codes
            extra = codes - reference_codes

            print(f"\n{name}")
            print("Missing compared with profiles:", len(missing))
            print("Extra compared with profiles:", len(extra))

            if missing:
                print("Missing stock codes:", sorted(missing)[:20])
            if extra:
                print("Extra stock codes:", sorted(extra)[:20])



Product profiles
File exists: True
Rows: 4,917
Unique non-null stock codes: 4,916
Missing stock codes: 0
Duplicate stock-code rows: 1

Demand trends
File exists: True
Rows: 4,917
Unique non-null stock codes: 4,916
Missing stock codes: 0
Duplicate stock-code rows: 1

Priority dataset
File exists: True
Rows: 4,917
Unique non-null stock codes: 4,916
Missing stock codes: 0
Duplicate stock-code rows: 1

Daily demand
File exists: True
Rows: 532,839
Unique non-null stock codes: 4,916
Missing stock codes: 0
Duplicate stock-code rows: 527,923

Product count comparison
Product profiles: 4,916 unique products
Demand trends: 4,916 unique products
Priority dataset: 4,916 unique products
Daily demand: 4,916 unique products

Differences compared with product profiles

Product profiles
Missing compared with profiles: 0
Extra compared with profiles: 0

Demand trends
Missing compared with profiles: 0
Extra compared with profiles: 0

Priority dataset
Missing compared with profiles: 0
Extra compared with